# TwoTank - Stage 3 - Nominal Controller Evaluation

Evaluate the configured controllers on matched scenarios.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "sdpc").is_dir() and (candidate / "case_studies").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SD-DPC repository")

REPO_ROOT = find_repo_root()
CASE_DIR = REPO_ROOT / "case_studies" / "TwoTank"
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import torch

from sdpc.config import load_config
from sdpc.eval import evaluate_twotank_nominal, policy_forward_latency
from sdpc.registry import make_system
from sdpc.sindy import load_model
from sdpc.training import load_nn_policy

device = torch.device('cpu')
system = make_system('twotank', device=device)
CONFIGS = CASE_DIR / 'configs'
RESULTS = CASE_DIR / 'results'
cfg = load_config(CONFIGS / 'eval_nominal.yaml')
print({k: cfg[k] for k in ('n_trajectories', 'nsteps', 'n_references')})
print('DPC-matched loss weights:', cfg['weights'])
print('Online MPC configuration:', cfg['mpc'])

## Load the two learned policies

Restore the required checkpoints.


In [ ]:
from sdpc.io import find_policy_checkpoint, find_nn_checkpoint
policy_path = find_policy_checkpoint(RESULTS, cfg)
sparse_policy = load_model(policy_path, device=device).eval()

nn_path = find_nn_checkpoint(RESULTS, cfg)
nn_policy = load_nn_policy(system, nn_path, cfg, device=device)
print('SD-DPC:', policy_path)
print('NN-DPC:', nn_path)

## Policy-only inference benchmark

Inspect policy performance.


In [ ]:
probe_x = torch.zeros(1, 1, system.nx, device=device)
probe_r = torch.zeros(1, 1, system.nx, device=device)
policy_latencies = {
    name: policy_forward_latency(
        policy, probe_x, probe_r, **cfg['inference_benchmark']
    )
    for name, policy in [('sd_dpc', sparse_policy), ('nn_dpc', nn_policy)]
}
policy_only = pd.DataFrame.from_dict(policy_latencies, orient='index')
policy_only.index.name = 'method'
policy_only['relative_to_sd'] = (
    policy_only['policy_forward_per_step_us']
    / policy_only.loc['sd_dpc', 'policy_forward_per_step_us']
)
display(policy_only[['policy_forward_per_step_us', 'policy_forward_per_step_ms', 'relative_to_sd']])

## Run the configured nominal trajectories

Run the matched nominal evaluation.


In [ ]:
nominal_out = evaluate_twotank_nominal(
    system, sparse_policy, nn_policy, cfg, device, progress=True,
    policy_latencies=policy_latencies,
)
scenario = nominal_out['scenarios']
assert scenario['xn'].shape == (cfg['n_trajectories'], 1, system.nx)
assert scenario['r'].shape == (cfg['n_trajectories'], cfg['nsteps'] + 1, system.nx)
assert torch.all(scenario['r'][..., 0] == scenario['r'][..., 1])
print('Distinct initial states:', torch.unique(scenario['xn'][:, 0], dim=0).shape[0])

## Aggregate metrics

Compare performance and runtime metrics.


In [ ]:
summary = pd.DataFrame.from_dict(nominal_out['summary'], orient='index')
summary.index.name = 'method'
summary_columns = [
    'tracking_mse_mean', 'tracking_mse_std',
    'control_smoothness_rms_mean', 'max_du_l2_mean',
    'num_state_violations_mean', 'num_control_rate_violations_mean',
    'min_safety_margin_mean', 'policy_forward_per_step_us_mean',
    'policy_forward_per_step_ms_mean',
    'mpc_solve_time_s_mean', 'mpc_solve_per_step_ms_mean',
    'mpc_setup_time_s_mean', 'mpc_iterations_per_solve_mean',
    'mpc_max_iterations_per_solve_mean', 'mpc_horizon_mean',
]
display(summary.reindex(columns=summary_columns))

## Violations summarized per trajectory

Visualize the closed-loop response.


In [ ]:
nominal_violations = pd.DataFrame.from_dict(nominal_out['violations'], orient='index')
nominal_violations.index.name = 'method'
display(nominal_violations)

## Per-trajectory results

Visualize the closed-loop response.


In [ ]:
nominal_rows = pd.DataFrame([
    row for rows in nominal_out['per_trajectory'].values() for row in rows
])
display(nominal_rows.head())

## Save tables

Save the generated artifacts.


In [ ]:
out_dir = RESULTS / 'eval' / f"nominal_{cfg['n_trajectories']}x{cfg['nsteps']}"
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'summary.csv')
nominal_violations.to_csv(out_dir / 'violation_summary.csv')
nominal_rows.to_csv(out_dir / 'per_trajectory.csv', index=False)
torch.save(
    {
        'trajectories': nominal_out['trajectories'],
        'scenarios': {
            key: value.detach().cpu() if isinstance(value, torch.Tensor) else value
            for key, value in nominal_out['scenarios'].items()
        },
    },
    out_dir / 'all_trajectories.pt',
)
print('Saved tables and all trajectories to', out_dir)
